# 03 — Multimodal Fusion + Multi-task (M4)
Tabular + IMT → Fusion → 3 head: plaque (BCE), echo (CE mask), risk (SmoothL1).
> Chạy cell setup trước. Cần GPU.

In [ ]:
SOURCE = "git"
GIT_URL = "https://github.com/<user>/Master-UIT-MedSignal.git"
DRIVE_PATH = "/content/drive/MyDrive/Master-UIT-MedSignal"
import os, sys
if SOURCE == "drive":
    from google.colab import drive; drive.mount('/content/drive'); PROJECT = DRIVE_PATH
else:
    PROJECT = "/content/Master-UIT-MedSignal"
    if not os.path.exists(PROJECT): os.system(f"git clone {GIT_URL} {PROJECT}")
os.chdir(PROJECT); sys.path.append(PROJECT)
!pip install -q -r requirements.txt

In [ ]:
# Kiem tra forward pass cua MultimodalFusion voi 1 batch that
import torch
from src.data import preprocess as P
from src.data.dataset import CarotidDataset, collate_fn
from src.data.splits import stratified_folds
from src.models.fusion import MultimodalFusion
from torch.utils.data import DataLoader

cfg = P.load_config("configs/config.yaml")
df = P.load_dataframe(cfg)
folds = stratified_folds(df, cfg)
tr, va = folds[0]
scaler = P.fit_scaler(P.encode_categorical(df.iloc[tr], cfg), cfg)
ds = CarotidDataset(df.iloc[tr], cfg, scaler)
batch = next(iter(DataLoader(ds, batch_size=8, collate_fn=collate_fn)))

model = MultimodalFusion(cfg, in_tab=len(P.feature_columns(cfg)))
out = model(batch['tabular'], batch['imt_img'], batch['cca_imgs'], batch['cca_mask'])
print({k: v.shape for k, v in out.items()})  # plaque[8,1] risk[8,1] echo[8,3]

In [ ]:
# Train end-to-end (KHUNG train.py chay 1 vong de smoke test).
# M4 TODO: bo dong `break` trong train_one_fold, them validate + early stopping + checkpoint.
!python -m src.train.train --config configs/config.yaml

**M4 TODO:** hoàn thiện vòng epoch trong [src/train/train.py](../src/train/train.py), thu thập metrics mỗi fold, lưu checkpoint tốt nhất (val PR-AUC) cho demo của M5; thử uncertainty weighting trong [src/train/losses.py](../src/train/losses.py).